Written with assistance from Claude (free tier).

# READ ME FIRST

- To toggle percentage or raw count plotting in the baseline / policy / reward function comparison plots, go to the cell defining USE_PERCENTAGES and set to True or False as desired.
- Plot formatting (fonts, font sizes, format_axes(), legend / axis-label placement) comes from Paper_results.ipynb. Data, experiments, colors and the fractional-blocking option come from Analysis.ipynb.
- MIN_MEASURED_DOMAINS (in the "Heat map" section) controls the gray "Fewer than N measured domains" cells in all heatmaps. Set it to 1 to only gray out untested cells.
- You should be able to change / add new plots just by changing the input files. This is vibecoded though, so can't make strong promises.

In [1]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import re
import textwrap


## Init

In [2]:
from math import sqrt

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ---- Formatting taken from Paper_results.ipynb ----
SPINE_COLOR = "#808080"
PAPER_FONT_SIZE = 8
PAPER_AXIS_LABEL_SIZE = 16
PAPER_TICK_LABEL_SIZE = 16
PAPER_LEGEND_SIZE = 16
PAPER_PANEL_LABEL_SIZE = 16

mpl.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman"],
    "font.size": PAPER_FONT_SIZE, "axes.labelsize": PAPER_AXIS_LABEL_SIZE,
    "axes.titlesize": PAPER_PANEL_LABEL_SIZE,
    "xtick.labelsize": PAPER_TICK_LABEL_SIZE, "ytick.labelsize": PAPER_TICK_LABEL_SIZE,
    "legend.fontsize": PAPER_LEGEND_SIZE,
    "axes.linewidth": 0.5, "axes.edgecolor": SPINE_COLOR,
    "xtick.direction": "in", "ytick.direction": "in",
    "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 300,
    "figure.facecolor": "white", "axes.facecolor": "white", "text.usetex": False,
})

def latexify(fig_width=None, fig_height=None, columns=1):
    """Set up matplotlib's RC params for LaTeX plotting.
    Call this before plotting a figure.
    Parameters
    ----------
    fig_width : float, optional, inches
    fig_height : float,  optional, inches
    columns : {1, 2}
    """

    # code adapted from http://www.scipy.org/Cookbook/Matplotlib/LaTeX_Examples

    # Width and max height in inches for IEEE journals taken from
    # computer.org/cms/Computer.org/Journal%20templates/transactions_art_guide.pdf

    assert(columns in [1,2])

    if fig_width is None:
        fig_width = 3.39 if columns==1 else 6.9 # width in inches

    if fig_height is None:
        golden_mean = (sqrt(5)-1.0)/2.0    # Aesthetic ratio
        fig_height = fig_width*golden_mean # height in inches

    MAX_HEIGHT_INCHES = 8.0
    if fig_height > MAX_HEIGHT_INCHES:
        print("WARNING: fig_height too large:" + fig_height +
              "so will reduce to" + MAX_HEIGHT_INCHES + "inches.")
        fig_height = MAX_HEIGHT_INCHES

    params = {'backend': 'ps',
              'text.latex.preamble': '\\usepackage{gensymb}',
              'axes.labelsize': 10, # fontsize for x and y labels (was 10)
              'axes.titlesize': 8,
              'font.size': 8, # was 10
              'legend.fontsize': 13, # was 10
              'xtick.labelsize': 8,
              'ytick.labelsize': 8,
              'text.usetex': True,
              'figure.figsize': [fig_width,fig_height],
              'font.family': 'times new roman'
    }

    mpl.rcParams.update(params)
    return params
    
# Lab axes format (from Paper_results.ipynb). Call on every axes.
def format_axes(ax):
    for spine in ['left', 'bottom', 'top', 'right']:
        ax.spines[spine].set_color(SPINE_COLOR)
        ax.spines[spine].set_linewidth(0.5)
    ax.tick_params(direction='in', color=SPINE_COLOR, labelsize=PAPER_TICK_LABEL_SIZE)
    ax.grid(True, linestyle=':', linewidth=0.8, color='#444444', alpha=0.35)
    return ax
    
COL_WIDTH = 3.39    # IEEE single-column width (inches)
FULL_WIDTH = 6.9    # IEEE full / two-column width (inches)
SINGLE_FIGSIZE = (COL_WIDTH, 2.4)   # one-country line plots

def tex(s):
    """With text.usetex on, escape characters LaTeX treats specially (& % $ # _) and turn × into $\\times$."""
    s = str(s)
    if not mpl.rcParams["text.usetex"]:
        return s
    for ch in "&%$#_":
        s = s.replace(ch, "\\" + ch)
    return s.replace("×", r"$\times$")

def bold(s):
    """\\textbf{} under usetex (fontweight is ignored there); unchanged otherwise."""
    return rf"\textbf{{{s}}}" if mpl.rcParams["text.usetex"] else s

def set_margins(fig, left, right, bottom, top):
    """Like fig.subplots_adjust, but margins are in INCHES, so spacing around 8 pt text
    stays the same whatever the figure size."""
    w, h = fig.get_size_inches()
    fig.subplots_adjust(left=left / w, right=1 - right / w, bottom=bottom / h, top=1 - top / h)
latexify()

{'backend': 'ps',
 'text.latex.preamble': '\\usepackage{gensymb}',
 'axes.labelsize': 10,
 'axes.titlesize': 8,
 'font.size': 8,
 'legend.fontsize': 13,
 'xtick.labelsize': 8,
 'ytick.labelsize': 8,
 'text.usetex': True,
 'figure.figsize': [3.39, 2.0951352218621437],
 'font.family': 'times new roman'}

## Load data

In [3]:
DATA_DIR = "data"
OUTPUT_DIR = "figures"

In [4]:
def load_run(path):
    df = pd.read_csv(path)
    # "categories Pornography" -> "Pornography"
    # the new fractional files only have category name and 
    if "category" not in df:
        df["category"] = df["action"].str.replace("categories ", "", regex=False)
    # i don't think the below is neceessary at all
    # return df.sort_values(["episode", "time"]).reset_index(drop=True)
    return df.sort_values(["time"]).reset_index(drop=True)

In [5]:
name_map = {
    "baseline": "Ranked baseline",
    "exp1": "Default (UCB)",
    "exp2": "LinUCB",
    "exp3": "Shared LinUCB",
    "exp4": "Cluster LinUCB",
    "exp5": "UCB, partial (VPs)",
    "exp6": "UCB, partial (ASNs)",
    "exp7": "UCB, maj. vote",
}

color_map = {
    "baseline": "grey",
    "exp1": "#4E79A7",
    "exp2": "green",
    "exp3": "purple",
    "exp4": "orange",
    "exp5": "#4E79A7",
    "exp6": "#4E79A7",
    "exp7": "#4E79A7",
}

line_map = {
    "baseline": "dashed",
    "exp1": "solid",
    "exp2": "dotted",
    "exp3": "dashdot",
    "exp4": "dashed",
    "exp5": "dotted",
    "exp6": "solid",
    "exp7": "dashed",
}

marker_map = {
    "baseline": "s",
    "exp1": "o",
    "exp2": "^",
    "exp3": "d",
    "exp4": "s",
    "exp5": "o",
    "exp6": "v",
    "exp7": "d",
}

def get_experiment_name(path):
    exp_num = None
    if "baseline" in path:
        exp_num = "baseline"
    else:
        exp_num = re.search(r"exp\d", path).group(0)
    return name_map[exp_num]

def get_experiment_color(path):
    exp_num = None
    if "baseline" in path:
        exp_num = "baseline"
    else:
        exp_num = re.search(r"exp\d", path).group(0)
    return color_map[exp_num]

def get_experiment_linestyle(path):
    exp_num = None
    if "baseline" in path:
        exp_num = "baseline"
    else:
        exp_num = re.search(r"exp\d", path).group(0)
    return line_map[exp_num]

def get_experiment_marker(path):
    exp_num = None
    if "baseline" in path:
        exp_num = "baseline"
    else:
        exp_num = re.search(r"exp\d", path).group(0)
    return marker_map[exp_num]


## Over-time comparisons (vs baseline, vs other policies, vs other rewards)

In [6]:
USE_PERCENTAGES = False
USE_FRACTIONAL_BLOCKING = True

In [7]:
def cumulative_found(df):
    if USE_FRACTIONAL_BLOCKING:
        # note we don't need to have special cases for baseline or majority vote, since
        # reward is 1.0 for blocking in those cases
        return df["rewards"].cumsum().to_numpy()
    return df["is_blocked"].cumsum().to_numpy()

In [8]:
def convert_to_percentage(run, total_blocked):
    return 100 * (run / total_blocked) if total_blocked > 0 else run
    # return 100 * (run / run[-1]) if run[-1] > 0 else run

In [9]:
def get_best_run_idx(runs):
    # print([run[-1] for run in runs])
    return np.argmax([run[-1] for run in runs])

In [10]:
from itertools import repeat

def get_over_time_compare_data(files):
    runs = list(map(load_run, files))
    times = list(map(lambda df: df["time"], runs))
    names = list(map(get_experiment_name, files))
    colors = list(map(get_experiment_color, files))
    linestyles = list(map(get_experiment_linestyle, files))
    markers = list(map(get_experiment_marker, files))

    if USE_PERCENTAGES:
        _ = pd.concat(runs)[["targets", "is_blocked"]]
        total_blocked = len(_[_["is_blocked"] == 1].drop_duplicates())
        # print(total_blocked)
        # for run in runs:
        #     print(len(run[["targets", "is_blocked"]].drop_duplicates())) 
        runs = list(map(convert_to_percentage, map(cumulative_found, runs), repeat(total_blocked)))
        # runs = list(map(convert_to_percentage, map(cumulative_found, runs)))
    else:
        runs = list(map(cumulative_found, runs))

    return runs, times, names, colors, linestyles, markers

In [11]:
def finish_comparison_plot(fig, ax, title):
    """Paper_results-style finishing for the over-time line plots:
    format_axes, legend above the axes, figure-level axis labels."""
    ax.set_title(tex(title))
    format_axes(ax)
    ax.set_axisbelow(True)

    # same logic as before, but as one label (supylabel can't be overwritten like set_ylabel)
    ylabel = "Num Censored Domains Found"
    if USE_PERCENTAGES:
        ylabel = "% of Censored Domains Found"
    if USE_FRACTIONAL_BLOCKING:
        ylabel = "Fractional Blocking Found"

    handles, labels = ax.get_legend_handles_labels()
    ncol = 2
    fig.legend(handles, [tex(l) for l in labels], ncol=ncol, loc="upper center", bbox_to_anchor=(.5, 1.0),
               frameon=True, edgecolor="#cccccc", borderpad=.3)
    fig.supxlabel("Number of Measurements", y=.02, fontsize=PAPER_AXIS_LABEL_SIZE)
    fig.supylabel(tex(ylabel), x=.006, fontsize=PAPER_AXIS_LABEL_SIZE)

    # margins in inches: room at the top for the legend (one row per `ncol` entries) and the title
    legend_rows = -(-len(labels) // ncol)
    set_margins(fig, left=.65, right=.1, bottom=.6, top=0.2 * legend_rows + 0.45)

def plot_comparison(files, country_code):
    runs, times, names, colors, lines, markers = get_over_time_compare_data(files) 
    best_run_idx = get_best_run_idx(runs)
    
    fig, ax = plt.subplots(figsize=SINGLE_FIGSIZE)
    for _ in range(len(runs)):
        lw = 0.8
        if "Baseline" in names[_] or _ == best_run_idx:
            lw = 1.8
        ax.plot(times[_], runs[_], label=names[_], color=colors[_], linestyle=lines[_], marker=markers[_], linewidth=lw)
    
    finish_comparison_plot(fig, ax, f"{country_code}")

In [12]:
from matplotlib.ticker import MaxNLocator, FixedLocator

def plot_comparison_panels(exps, ncol=None, figsize=(FULL_WIDTH, 2.4)):
    """China / Iran / UAE side by side in ONE figure.
    exps: experiment folder names, in plotting order, e.g. ["exp1", "baseline"]."""
    fig, axes = plt.subplots(1, len(COMPARISON_COUNTRIES), figsize=figsize)

    
    for panel, (ax, (country_file, country_title)) in enumerate(zip(axes, COMPARISON_COUNTRIES)):
        files = [f"{DATA_DIR}/{exp}/{country_file}_fractional.csv" if USE_FRACTIONAL_BLOCKING and (exp == "baseline" or exp == "exp7")
                 else f"{DATA_DIR}/{exp}/{country_file}.csv"
                 for exp in exps]
            
        runs, times, names, colors, lines, markers = get_over_time_compare_data(files)
        # for _ in range(len(runs)):

        # this converts to fractional blocking %
        runs = [run / 2000 * 100 for run in runs]
        
        # best_run_idx = get_best_run_idx(runs)
        for _ in range(len(runs)):
            print((country_title, names[_]))
            print([runs[_][0], runs[_][499], runs[_][999], runs[_][1499], runs[_][1999]])
            
            lw = 1.4
            # if "Baseline" in names[_] or _ == best_run_idx:
            #     lw = 1.8
            ax.plot(times[_], runs[_], label=names[_], color=colors[_],
                    linestyle=lines[_], marker=markers[_], 
                    markevery=500, markersize=3, 
                    markerfacecolor="white", markeredgewidth=0.7, linewidth=lw)

        ax.set_title(f"({chr(97 + panel)}) {country_title}", loc="left", pad=2, fontsize=12)
        # at least 3 y ticks (0 plus two more), at most 4
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3, min_n_ticks=3, integer=True, prune=None))
        # also set min_n_ticks on x-axis
        ax.xaxis.set_major_locator(FixedLocator([0, 1000, 2000]))
        format_axes(ax)
        if len(COMPARISON_COUNTRIES) > 3:
            ax.tick_params(axis="y", labelsize=12)
            ax.tick_params(axis="x", labelsize=12)
        ax.set_axisbelow(True)
        ax.set_axisbelow(True)

    # same y-label logic as finish_comparison_plot
    ylabel = "# Blocked Domains"
    if USE_PERCENTAGES:
        ylabel = "% of Censored Domains Found"
    if USE_FRACTIONAL_BLOCKING:
        ylabel = "Fractional Blocking %"
    
    handles, labels = axes[0].get_legend_handles_labels()
    if ncol is None:
        ncol = min(len(labels), 5)   # one row unless there are many entries
    fig.legend(handles, [tex(l) for l in labels], ncol=ncol, loc="upper center", bbox_to_anchor=(.5, 1.0),
               frameon=True, edgecolor="#cccccc", borderpad=.3,
               columnspacing=1.2, handlelength=2.0)
    h = fig.get_figheight()
    fig.supxlabel("Number of Measurements", y=.04 / h, fontsize=PAPER_AXIS_LABEL_SIZE)
    fig.supylabel(tex(ylabel), x=.006, fontsize=PAPER_AXIS_LABEL_SIZE)

    # margins in inches: legend rows + one title line above the axes
    legend_rows = -(-len(labels) // ncol)
    set_margins(fig, left=.7, right=.1, bottom=.55, top=0.25 * legend_rows + 0.35)
    fig.subplots_adjust(wspace=.3)
    return fig, axes

### vs baseline

In [13]:
COMPARE_BASELINE_DIR = "./figures/compare_baseline"

In [14]:
# USE_FRACTIONAL_BLOCKING = False
COMPARISON_COUNTRIES = [("China", "China"), ("Iran", "Iran"), ("United_Arab_Emirates", "UAE")]
# COMPARISON_COUNTRIES = [("China", "China"), ("Iran", "Iran"), ("United_Arab_Emirates", "UAE"), ("Germany", "Germany"), ("Thailand", "Thailand"), ("Turkey", "Turkey"), ("Vietnam", "Vietnam")]
plot_comparison_panels(["exp1", "baseline"])
plt.savefig(f"{COMPARE_BASELINE_DIR}/results_default_baseline.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

('China', 'Default (UCB)')
[np.float64(0.0), np.float64(5.269499999999998), np.float64(10.718500000000002), np.float64(12.531999999999972), np.float64(12.736999999999973)]
('China', 'Ranked baseline')
[np.float64(5.5e-05), np.float64(2.6235650000000024), np.float64(4.471340000000003), np.float64(6.628245000000009), np.float64(8.894810000000021)]
('Iran', 'Default (UCB)')
[np.float64(0.046), np.float64(17.78649999999999), np.float64(32.45949999999995), np.float64(37.808999999999905), np.float64(40.33999999999987)]
('Iran', 'Ranked baseline')
[np.float64(0.0), np.float64(3.2468749999999997), np.float64(6.278124999999999), np.float64(9.165624999999999), np.float64(16.04450500000001)]
('UAE', 'Default (UCB)')
[np.float64(0.0), np.float64(4.797500000000006), np.float64(6.121000000000004), np.float64(6.759000000000012), np.float64(6.974000000000015)]
('UAE', 'Ranked baseline')
[np.float64(0.0), np.float64(0.2630149999999998), np.float64(0.5369699999999992), np.float64(0.8393849999999987), np

/tmp/ipykernel_56799/571298341.py:6: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
COMPARISON_COUNTRIES = [("Germany", "Germany"), ("Thailand", "Thailand"), ("Turkey", "Turkey"), ("Vietnam", "Vietnam")]
plot_comparison_panels(["exp1", "baseline"])
plt.savefig(f"{COMPARE_BASELINE_DIR}/results_default_baseline_others.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

('Germany', 'Default (UCB)')
[np.float64(0.0), np.float64(0.06950000000000005), np.float64(0.16549999999999876), np.float64(0.1959999999999982), np.float64(0.206999999999998)]
('Germany', 'Ranked baseline')
[np.float64(9.5e-05), np.float64(0.0881299999999998), np.float64(0.15136000000000083), np.float64(0.20660500000000215), np.float64(0.28254000000000157)]
('Thailand', 'Default (UCB)')
[np.float64(0.0), np.float64(0.09000000000000008), np.float64(0.15349999999999905), np.float64(0.17799999999999866), np.float64(0.20499999999999832)]
('Thailand', 'Ranked baseline')
[np.float64(0.0), np.float64(0.20605999999999952), np.float64(0.32309500000000024), np.float64(0.4452600000000009), np.float64(0.55752)]
('Turkey', 'Default (UCB)')
[np.float64(0.0), np.float64(0.7495000000000026), np.float64(0.9275000000000043), np.float64(1.0960000000000067), np.float64(1.1940000000000088)]
('Turkey', 'Ranked baseline')
[np.float64(0.0), np.float64(0.24451000000000017), np.float64(0.4276600000000008), np.f

/tmp/ipykernel_56799/1542238239.py:4: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


### Policy comparison

In [16]:
COMPARE_POLICY_DIR = "./figures/compare_policy"

In [17]:
COMPARISON_COUNTRIES = [("China", "China"), ("Iran", "Iran"), ("United_Arab_Emirates", "UAE")]
plot_comparison_panels(["exp1", "exp2", "exp3", "exp4"], ncol=2)
# plot_comparison_panels(["exp1", "exp2", "exp3", "exp4", "baseline"])
plt.savefig(f"{COMPARE_POLICY_DIR}/results_policy_comparison.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

('China', 'Default (UCB)')
[np.float64(0.0), np.float64(5.269499999999998), np.float64(10.718500000000002), np.float64(12.531999999999972), np.float64(12.736999999999973)]
('China', 'LinUCB')
[np.float64(0.0005), np.float64(2.0315000000000007), np.float64(6.114500000000005), np.float64(10.973499999999973), np.float64(13.355999999999952)]
('China', 'Shared LinUCB')
[np.float64(0.0), np.float64(4.483999999999998), np.float64(8.424500000000004), np.float64(11.148500000000018), np.float64(13.024500000000023)]
('China', 'Cluster LinUCB')
[np.float64(0.0), np.float64(2.271500000000002), np.float64(6.756000000000014), np.float64(10.082500000000017), np.float64(14.23649999999995)]
('Iran', 'Default (UCB)')
[np.float64(0.046), np.float64(17.78649999999999), np.float64(32.45949999999995), np.float64(37.808999999999905), np.float64(40.33999999999987)]
('Iran', 'LinUCB')
[np.float64(0.0), np.float64(8.225999999999996), np.float64(25.84800000000016), np.float64(34.44800000000005), np.float64(41.387

/tmp/ipykernel_56799/168386065.py:5: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
COMPARISON_COUNTRIES = [("Germany", "Germany"), ("Thailand", "Thailand"), ("Turkey", "Turkey"), ("Vietnam", "Vietnam")]
plot_comparison_panels(["exp1", "exp2", "exp3", "exp4"], ncol=2)
# plot_comparison_panels(["exp1", "exp2", "exp3", "exp4", "baseline"])
plt.savefig(f"{COMPARE_POLICY_DIR}/results_policy_comparison_others.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

('Germany', 'Default (UCB)')
[np.float64(0.0), np.float64(0.06950000000000005), np.float64(0.16549999999999876), np.float64(0.1959999999999982), np.float64(0.206999999999998)]
('Germany', 'LinUCB')
[np.float64(0.0), np.float64(0.07500000000000005), np.float64(0.1444999999999999), np.float64(0.21949999999999942), np.float64(0.3484999999999968)]
('Germany', 'Shared LinUCB')
[np.float64(0.0), np.float64(0.15499999999999975), np.float64(0.3394999999999975), np.float64(0.5064999999999947), np.float64(0.6149999999999923)]
('Germany', 'Cluster LinUCB')
[np.float64(0.0), np.float64(0.04850000000000002), np.float64(0.12550000000000003), np.float64(0.23899999999999924), np.float64(0.3894999999999961)]
('Thailand', 'Default (UCB)')
[np.float64(0.0), np.float64(0.09000000000000008), np.float64(0.15349999999999905), np.float64(0.17799999999999866), np.float64(0.20499999999999832)]
('Thailand', 'LinUCB')
[np.float64(0.0014999999999999998), np.float64(0.09450000000000007), np.float64(0.21799999999999

/tmp/ipykernel_56799/3706561741.py:5: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


### Reward comparison

In [19]:
COMPARE_REWARD_DIR = "./figures/compare_reward"

In [20]:
plot_comparison_panels(["exp5", "exp6", "exp7", "baseline"])
plt.savefig(f"{COMPARE_REWARD_DIR}/countries.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

('Germany', 'UCB, partial (VPs)')
[np.float64(0.0), np.float64(0.06950000000000005), np.float64(0.16549999999999876), np.float64(0.1959999999999982), np.float64(0.206999999999998)]
('Germany', 'UCB, partial (ASNs)')
[np.float64(0.001), np.float64(0.2674999999999995), np.float64(0.32549999999999885), np.float64(0.4059999999999977), np.float64(0.4424999999999969)]
('Germany', 'UCB, maj. vote')
[np.float64(0.0), np.float64(0.06248499999999983), np.float64(0.14057499999999956), np.float64(0.19959999999999922), np.float64(0.26088000000000083)]
('Germany', 'Ranked baseline')
[np.float64(9.5e-05), np.float64(0.0881299999999998), np.float64(0.15136000000000083), np.float64(0.20660500000000215), np.float64(0.28254000000000157)]
('Thailand', 'UCB, partial (VPs)')
[np.float64(0.0), np.float64(0.09000000000000008), np.float64(0.15349999999999905), np.float64(0.17799999999999866), np.float64(0.20499999999999832)]
('Thailand', 'UCB, partial (ASNs)')
[np.float64(0.0), np.float64(0.08150000000000006),

/tmp/ipykernel_56799/3763662595.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


## Box-whisker plot of arm pulls

In [21]:
def get_categories_to_plot(runs, min_domains, num_each):
    # print(runs)
    runs = map(load_run, runs)

    # _ = list(runs)[1]
    # print(_[_["category"] == "News & Media"])
    
    all_measured_domains = pd.concat(runs).drop_duplicates(subset="targets")
    filtered_domains = all_measured_domains.groupby("category").agg(
        n_domains=("targets", "nunique"),
        avg_reward=("rewards", "mean"),
    )
    filtered_domains = filtered_domains[filtered_domains["n_domains"] >= min_domains]
    filtered_domains = filtered_domains.sort_values("avg_reward", ascending=False)
    print(filtered_domains)

    top = list(filtered_domains.index[:num_each])
    bottom = list(filtered_domains.index[-num_each:])
    # dict.fromkeys drops repeats but keeps the order
    selected = list(dict.fromkeys(top + bottom))

    return selected

In [22]:
def get_scatter_plot_data(files, categories):
    runs = map(load_run, files)
    all_categories_per_run = [
        {cat: run[run["category"] == cat] for cat in categories}
        for run in runs
    ]
    category_rewards_per_run = [
        {cat: subs[cat]["rewards"].mean() for cat in categories}
        for subs in all_categories_per_run
    ]
    
    names = list(map(get_experiment_name, files))
    colors = list(map(get_experiment_color, files))
    
    return list(all_categories_per_run), list(category_rewards_per_run), names, colors

In [23]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import to_rgba

MIN_ALPHA = 0.1   # opacity at reward = 0 (keeps zero-reward points faintly visible)
MAX_ALPHA = 1.0   # opacity at reward = 1

def strength_to_colors(strength, base="crimson"):
    s = np.clip(np.asarray(strength, dtype=float), 0, 1)   # keep values in 0..1
    rgba = np.tile(to_rgba(base), (len(s), 1))              # one crimson row per point
    rgba[:, 3] = MIN_ALPHA + (MAX_ALPHA - MIN_ALPHA) * s    # overwrite the alpha column
    return rgba
    
from matplotlib.patches import Patch, Rectangle
from matplotlib.legend_handler import HandlerBase

class GradientKey:
    """Placeholder object; the handler below draws the strip for it."""

class GradientHandler(HandlerBase):
    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        n = 20
        w = width / n
        return [Rectangle((xdescent + k * w, ydescent), w, height, linewidth=0,
                          facecolor=to_rgba("crimson",
                                            MIN_ALPHA + (MAX_ALPHA - MIN_ALPHA) * k / (n - 1)),
                          transform=trans)
                for k in range(n)]

# Box colors for the scatter/box plots (palette from Paper_results.ipynb).
# Keyed like color_map; exp1 gets Paper's "Default (UCB)" blue.
box_color_map = {
    "baseline": "#777777",
    "exp1": "#4E79A7",
    "exp2": "#F28E2B",
    "exp3": "#59A14F",
    "exp4": "#E15759",
    "exp5": "#4E79A7",
    "exp6": "#B07AA1",
    "exp7": "#9C755F",
}

def get_box_color(path):
    exp_key = "baseline" if "baseline" in path else re.search(r"exp\d", path).group(0)
    return box_color_map[exp_key]

def plot_scatter_boxes(files, min_domains, n_each, country_code):
    # pop, readd is to avoid including baseline in get_categories_to_plot
    # Don't include baseline when calculating what categories to plot,
    # since we want to pick the most interesting categories reward-wise
    # and baseline doesn't try to pick "interesting" categories
    baseline, baseline_idx = None, None
    baseline_matches = [_ for _ in files if "baseline" in _]
    if baseline_matches:
        baseline_idx = files.index(baseline_matches[0])
        baseline = files.pop(baseline_idx)
    # _ = files.pop(0)
    categories = get_categories_to_plot(files, min_domains, n_each)
    # files.insert(0, _)
    if baseline_matches:
        files.append(baseline)
    
    X_AXIS_OFFSET_SCALE = 1
    BETWEEN_BOX_OFFSET_SCALE = 0.4
    PLOT_LENGTH_SCALE = 0.5
    BOX_WIDTH = 0.26
    
    def box_x(i, offset):
        return X_AXIS_OFFSET_SCALE * i + offset * BETWEEN_BOX_OFFSET_SCALE
    
    num_categories = len(categories)
    num_runs = len(files)
    fig, ax = plt.subplots(figsize=(FULL_WIDTH, 2.8))
    all_categories, category_rewards, names, colors = get_scatter_plot_data(files, categories)
    colors = [get_box_color(f) for f in files]   # box colors from Paper_results palette
    
    for i, cat in enumerate(categories):
        for offset, subs, color in zip(range(len(all_categories)), all_categories, colors):
            data = subs[cat]["time"].to_numpy()
            bp = ax.boxplot([data], 
                            positions=[box_x(i, offset)], 
                            widths=BOX_WIDTH,
                            patch_artist=True, showfliers=False,
                            boxprops={"facecolor": color, "edgecolor": color, "alpha": .28, "linewidth": .7},
                            medianprops={"color": "black", "linewidth": .7},
                            whiskerprops={"color": color, "linewidth": .5},
                            capprops={"color": color, "linewidth": .5})
    
    rng = np.random.default_rng(0)  # fixed seed -> same jitter every run
    
    JITTER_HALF_LENGTH = BOX_WIDTH / 3
    for i, cat in enumerate(categories):
        for offset, subs in zip(range(len(all_categories)), all_categories):
            pts = subs[cat]
            # spread points sideways around the box center (box half-width is 0.5)
            x = box_x(i, offset) + rng.uniform(-JITTER_HALF_LENGTH, JITTER_HALF_LENGTH, size=len(pts))
            if USE_FRACTIONAL_BLOCKING:
                # rewards = np.clip(pts["rewards"].to_numpy(), 0, 1)
                # alphas=rewards
                # point_colors = reward_cmap(rewards)
                strength = strength_to_colors(pts["rewards"])
                print(f"{cat}, {files[offset]}")
                print(len(pts["rewards"]))
                print(np.sum(pts["rewards"]))
            else:
                # point_colors = np.where(pts["is_blocked"] == 1, "crimson", "gray")
                # alphas = point_colors
                strength = pts["is_blocked"]
            # ax.scatter(x, pts["time"], c="crimson", s=6, alpha=alphas, zorder=3)
            # ax.scatter(x, pts["time"], c=point_colors, s=6, alpha=0.5, zorder=3)
            ax.scatter(x, pts["time"], c=strength, s=4, zorder=3)

    # y axis: ticks at 0 / half / max step (these give the dotted gridlines), with room on top for the counts
    max_step = max(subs[cat]["time"].max() for subs in all_categories for cat in categories)
    yticks = [0, max_step // 2, max_step]
    ax.set(ylim=(0, max_step * 1.12), yticks=yticks)
    ax.set_yticklabels([f"{t:,}" for t in yticks])

    # total (fractional) blocking found per box, printed above it in the box's color
    for i, cat in enumerate(categories):
        for offset, subs, color in zip(range(len(all_categories)), all_categories, colors):
            pts = subs[cat]
            if USE_FRACTIONAL_BLOCKING:
                total_text = f"{pts['rewards'].sum():.2f}"
            else:
                total_text = f"{int(pts['is_blocked'].sum())}"
            ax.text(box_x(i, offset), max_step * 1.065, bold(total_text), ha="center", va="center",
                    fontsize=12, color=color, fontweight="bold")

    # x tick labels: just the category name (average reward removed)
    labels = [tex(textwrap.fill(cat, width=14, break_long_words=False)) for cat in categories]
    
    ax.set_xticks([box_x(i, (num_runs - 1) / 2) for i in range(len(categories))])
    ax.set_xticklabels(labels)
    
    ax.set_ylabel("Measurement Step", fontsize=16)
    # ax.set_title(f"{country_code}", pad=22)

    # handles = [Patch(color=color) for color in colors] + [GradientKey()]
    # labels = list(names) + ["Fraction of vantage points\nblocking (faint = 0, solid = 1)"]
    handles = [Patch(color=color) for color in colors]
    labels = [tex(n) for n in names]
    
    ax.legend(handles=handles, labels=labels,
               handler_map={GradientKey: GradientHandler()},
               handlelength=2.5,   # a wider handle makes the gradient easier to see
               ncol=2, loc="lower center", bbox_to_anchor=(.5, 0.9), borderaxespad=1.8,
               frameon=True, edgecolor="#cccccc", borderpad=.3,
               columnspacing=1.2)
    format_axes(ax)
    # ax.tick_params(axis="x", labelsize=12)   # after format_axes, which sets 16
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)
    # fig.subplots_adjust(left=.13, right=.99, bottom=.19, top=.66)
    set_margins(fig, left=.6, right=.1, bottom=.42, top=.8)


In [24]:
SCATTER_DIR = "./figures/category_times"

In [25]:
min_domains_Iran = 100   # category must have at least this many distinct domains
n_each_Iran = 3          # how many to take from the top and from the bottom

files_Iran = [
    f"{DATA_DIR}/exp1/Iran.csv",
    # f"{DATA_DIR}/exp3/Iran.csv",
    f"{DATA_DIR}/baseline/Iran_fractional.csv"
]

plot_scatter_boxes(files_Iran, min_domains_Iran, n_each_Iran, "IR")
plt.savefig(f"{SCATTER_DIR}/Iran_baseline.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                      n_domains  avg_reward
category                                   
Pornography                 329    0.926322
Social Networks             239    0.797741
Personal Blogs              114    0.286053
Ecommerce                   421    0.137696
Advertisements              132    0.125606
Information Security        103    0.019417
Pornography, data/exp1/Iran.csv
329
304.76
Pornography, data/baseline/Iran_fractional.csv
83
62.1236
Social Networks, data/exp1/Iran.csv
239
190.66
Social Networks, data/baseline/Iran_fractional.csv
41
27.023199999999996
Personal Blogs, data/exp1/Iran.csv
114
32.61
Personal Blogs, data/baseline/Iran_fractional.csv
15
7.827399999999999
Ecommerce, data/exp1/Iran.csv
421
57.97
Ecommerce, data/baseline/Iran_fractional.csv
83
9.6292
Advertisements, data/exp1/Iran.csv
132
16.58
Advertisements, data/baseline/Iran_fractional.csv
35
8.4375
Information Security, data/exp1/Iran.csv
103
2.0
Information Security, data/baseline/Iran_fractional.csv
29
0.0

/tmp/ipykernel_56799/633801372.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [26]:
min_domains_Iran = 10   # category must have at least this many distinct domains
n_each_Iran = 20          # how many to take from the top and from the bottom

files_Iran = [
    f"{DATA_DIR}/exp1/Iran.csv",
    f"{DATA_DIR}/exp4/Iran.csv",
    # f"{DATA_DIR}/baseline/Iran.csv"
]

plot_scatter_boxes(files_Iran, min_domains_Iran, n_each_Iran, "IR")
plt.savefig(f"{SCATTER_DIR}/Iran_policy.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                                            n_domains  avg_reward
category                                                         
Pornography                                       329    0.926322
Social Networks                                   239    0.797741
Dating & Relationships                             17    0.734118
P2P                                                11    0.698182
News, Portal & Search                              10    0.643000
Adult Themes                                       32    0.599375
Redirect                                           17    0.573529
Cartoons & Anime                                   20    0.517000
Movies                                             84    0.491190
Comic Books                                        21    0.444762
Instant Messengers                                 48    0.430625
Video Streaming                                   392    0.396531
Music                                              22    0.377273
Forums    

/tmp/ipykernel_56799/86279393.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [27]:
min_domains_Iran = 10   # category must have at least this many distinct domains
n_each_Iran = 3          # how many to take from the top and from the bottom

files_Iran = [
    f"{DATA_DIR}/exp5/Iran.csv",
    f"{DATA_DIR}/exp6/Iran.csv",
    # f"{DATA_DIR}/baseline/Iran.csv"
]

plot_scatter_boxes(files_Iran, min_domains_Iran, n_each_Iran, "Iran")
plt.savefig(f"{SCATTER_DIR}/Iran_reward.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                         n_domains  avg_reward
category                                      
Pornography                    329    0.926322
Social Networks                239    0.797741
Dating & Relationships          17    0.727647
P2P                             11    0.698182
News, Portal & Search           10    0.643000
Adult Themes                    32    0.599375
Redirect                        17    0.573529
Cartoons & Anime                20    0.517000
Movies                          84    0.491190
Comic Books                     21    0.444762
Photography                     37    0.432162
Instant Messengers              48    0.430625
Unreachable                     14    0.422857
Entertainment                   52    0.405000
Gambling                        88    0.398864
Video Streaming                391    0.376394
Forums                          66    0.346212
Television                      74    0.323514
Audio Streaming                 50    0.318600
Internet Phon

/tmp/ipykernel_56799/725447287.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


### China

In [28]:
min_domains_China = 100   # category must have at least this many distinct domains
n_each_China = 3          # how many to take from the top and from the bottom

files_China = [
    f"{DATA_DIR}/exp1/China.csv",
    # f"{DATA_DIR}/exp3/China.csv",
    f"{DATA_DIR}/baseline/China_fractional.csv"
]

plot_scatter_boxes(files_China, min_domains_China, n_each_China, "China")
plt.savefig(f"{SCATTER_DIR}/results_category_timing.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                 n_domains  avg_reward
category                              
Video Streaming        535    0.211252
Pornography            326    0.073865
Technology             522    0.009789
Video Streaming, data/exp1/China.csv
535
113.01999999999998
Video Streaming, data/baseline/China_fractional.csv
51
16.107199999999995
Pornography, data/exp1/China.csv
326
24.080000000000002
Pornography, data/baseline/China_fractional.csv
83
9.4195
Technology, data/exp1/China.csv
522
5.11
Technology, data/baseline/China_fractional.csv
183
8.338799999999999


/tmp/ipykernel_56799/108031092.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [29]:
min_domains_China = 100   # category must have at least this many distinct domains
n_each_China = 3          # how many to take from the top and from the bottom

files_China = [
    f"{DATA_DIR}/exp1/China.csv",
    f"{DATA_DIR}/exp2/China.csv",
    # f"{DATA_DIR}/baseline/China.csv"
]

plot_scatter_boxes(files_China, min_domains_China, n_each_China, "China")
plt.savefig(f"{SCATTER_DIR}/China_policy.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                 n_domains  avg_reward
category                              
Video Streaming        535    0.211252
News & Media           110    0.109091
Pornography            326    0.073865
Technology             525    0.009733
Video Streaming, data/exp1/China.csv
535
113.01999999999998
Video Streaming, data/exp2/China.csv
498
107.3
News & Media, data/exp1/China.csv
1
0.0
News & Media, data/exp2/China.csv
127
15.909999999999998
Pornography, data/exp1/China.csv
326
24.080000000000002
Pornography, data/exp2/China.csv
43
3.4299999999999997
Technology, data/exp1/China.csv
522
5.11
Technology, data/exp2/China.csv
10
0.0


/tmp/ipykernel_56799/3077358938.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
min_domains_China = 100   # category must have at least this many distinct domains
n_each_China = 3          # how many to take from the top and from the bottom

files_China = [
    f"{DATA_DIR}/exp5/China.csv",
    f"{DATA_DIR}/exp7/China.csv",
    # f"{DATA_DIR}/baseline/China.csv"
]

plot_scatter_boxes(files_China, min_domains_China, n_each_China, "China")
plt.savefig(f"{SCATTER_DIR}/China_reward.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                        n_domains  avg_reward
category                                     
Video Streaming               535    0.211252
News & Media                  623    0.128411
Pornography                   326    0.073865
Ecommerce                     399    0.045113
Information Technology        214    0.023551
Technology                    523    0.009771
Video Streaming, data/exp5/China.csv
535
113.01999999999998
Video Streaming, data/exp7/China.csv
2
0.0
News & Media, data/exp5/China.csv
1
0.0
News & Media, data/exp7/China.csv
908
141.0
Pornography, data/exp5/China.csv
326
24.080000000000002
Pornography, data/exp7/China.csv
1
0.0
Ecommerce, data/exp5/China.csv
1
0.0
Ecommerce, data/exp7/China.csv
416
18.0
Information Technology, data/exp5/China.csv
16
0.04
Information Technology, data/exp7/China.csv
217
5.0
Technology, data/exp5/China.csv
522
5.11
Technology, data/exp7/China.csv
1
0.0


/tmp/ipykernel_56799/2645642294.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


### UAE

I expect this to look the most interesting when comparing majority vs partial votes?

In [31]:
min_domains_UAE = 100   # category must have at least this many distinct domains
n_each_UAE = 3          # how many to take from the top and from the bottom

files_UAE = [
    f"{DATA_DIR}/exp1/United_Arab_Emirates.csv",
    # f"{DATA_DIR}/exp7/United_Arab_Emirates.csv",
    f"{DATA_DIR}/baseline/China.csv"
]

plot_scatter_boxes(files_UAE, min_domains_UAE, n_each_UAE, "United_Arab_Emirates")
plt.savefig(f"{SCATTER_DIR}/UAE_baseline.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                 n_domains  avg_reward
category                              
Pornography            327    0.259878
Social Networks        198    0.026818
Sports                 129    0.010388
Search Engines         348    0.007241
Pornography, data/exp1/United_Arab_Emirates.csv
327
84.97999999999999
Pornography, data/baseline/China.csv
83
9.0
Social Networks, data/exp1/United_Arab_Emirates.csv
198
5.31
Social Networks, data/baseline/China.csv
41
7.0
Sports, data/exp1/United_Arab_Emirates.csv
129
1.34
Sports, data/baseline/China.csv
7
0.0
Search Engines, data/exp1/United_Arab_Emirates.csv
348
2.52
Search Engines, data/baseline/China.csv
61
3.0


/tmp/ipykernel_56799/248135742.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
min_domains_UAE = 100   # category must have at least this many distinct domains
n_each_UAE = 3          # how many to take from the top and from the bottom

files_UAE = [
    f"{DATA_DIR}/exp1/United_Arab_Emirates.csv",
    f"{DATA_DIR}/exp2/United_Arab_Emirates.csv",
    # f"{DATA_DIR}/baseline/China.csv"
]

plot_scatter_boxes(files_UAE, min_domains_UAE, n_each_UAE, "United_Arab_Emirates")
plt.savefig(f"{SCATTER_DIR}/UAE_policy.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                 n_domains  avg_reward
category                              
Pornography            327    0.259878
Social Networks        198    0.026818
Sports                 129    0.010388
Search Engines         348    0.007241
Pornography, data/exp1/United_Arab_Emirates.csv
327
84.97999999999999
Pornography, data/exp2/United_Arab_Emirates.csv
325
85.72
Social Networks, data/exp1/United_Arab_Emirates.csv
198
5.31
Social Networks, data/exp2/United_Arab_Emirates.csv
86
2.7599999999999993
Sports, data/exp1/United_Arab_Emirates.csv
129
1.34
Sports, data/exp2/United_Arab_Emirates.csv
35
1.5299999999999998
Search Engines, data/exp1/United_Arab_Emirates.csv
348
2.52
Search Engines, data/exp2/United_Arab_Emirates.csv
21
0.06


/tmp/ipykernel_56799/3329707055.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [33]:
min_domains_UAE = 100   # category must have at least this many distinct domains
n_each_UAE = 3          # how many to take from the top and from the bottom

files_UAE = [
    f"{DATA_DIR}/exp5/United_Arab_Emirates.csv",
    f"{DATA_DIR}/exp6/United_Arab_Emirates.csv",
    # f"{DATA_DIR}/baseline/China.csv"
]

plot_scatter_boxes(files_UAE, min_domains_UAE, n_each_UAE, "United_Arab_Emirates")
plt.savefig(f"{SCATTER_DIR}/UAE_reward.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

                 n_domains  avg_reward
category                              
Pornography            327    0.259878
Social Networks        198    0.026818
Sports                 129    0.010388
Advertisements         127    0.008268
Search Engines         348    0.007241
Education              427    0.003794
Pornography, data/exp5/United_Arab_Emirates.csv
327
84.97999999999999
Pornography, data/exp6/United_Arab_Emirates.csv
329
41.2
Social Networks, data/exp5/United_Arab_Emirates.csv
198
5.31
Social Networks, data/exp6/United_Arab_Emirates.csv
238
11.96
Sports, data/exp5/United_Arab_Emirates.csv
129
1.34
Sports, data/exp6/United_Arab_Emirates.csv
1
0.0
Advertisements, data/exp5/United_Arab_Emirates.csv
4
0.0
Advertisements, data/exp6/United_Arab_Emirates.csv
134
1.29
Search Engines, data/exp5/United_Arab_Emirates.csv
348
2.52
Search Engines, data/exp6/United_Arab_Emirates.csv
1
0.0
Education, data/exp5/United_Arab_Emirates.csv
36
0.09
Education, data/exp6/United_Arab_Emirates.csv
449

/tmp/ipykernel_56799/833991872.py:12: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [34]:
# all_categories, category_rewards, names, colors = get_scatter_plot_data(files, categories)

# print(len(all_categories), names)                        # one entry per run?
# for subs in all_categories:
#     print({c: len(subs[c]) for c in categories})         # rows per category
# print(all_categories[0][categories[0]].columns.tolist()) # expected column names?

In [35]:
# exp_sub = {cat: exp[exp["category"] == cat] for cat in selected}
# base_sub = {cat: base[base["category"] == cat] for cat in selected}

In [36]:
# from matplotlib.lines import Line2D
# from matplotlib.patches import Patch

# X_AXIS_OFFSET_SCALE = 1
# BETWEEN_BOX_OFFSET_SCALE = 0.3
# PLOT_LENGTH_SCALE = 0.5
# BOX_WIDTH = 0.2

# def box_x(i, offset):
#     return X_AXIS_OFFSET_SCALE * i + offset * BETWEEN_BOX_OFFSET_SCALE

# # plot length should be dependent on # categories and # runs
# num_categories = len(categories)
# num_runs = len(files)

# # fig, ax = plt.subplots(figsize=(num_categories * num_runs * PLOT_LENGTH_SCALE, 6))
# fig, ax = plt.subplots(figsize=(12, 6))

# # need a list of idx, run subsets, rewards, names, colors
# # run subsets are the measurements that match each category for every cresultsategory, so actually a 

# all_categories, category_rewards, names, colors = get_scatter_plot_data(files, categories)

# for i, cat in enumerate(categories):
#     for offset, subs, color in zip(range(len(all_categories)), all_categories, colors):
#         data = subs[cat]["time"].to_numpy()
#         bp = ax.boxplot([data], 
#                         positions=[box_x(i, offset)], 
#                         widths=BOX_WIDTH,
#                         patch_artist=True, showfliers=False,
#                         medianprops={"color": "black"})
#         bp["boxes"][0].set_facecolor(color)
#         bp["boxes"][0].set_alpha(0.4)   # make the box semi-transparent

# rng = np.random.default_rng(0)  # fixed seed -> same jitter every run

# JITTER_HALF_LENGTH = BOX_WIDTH / 3
# for i, cat in enumerate(categories):
#     for offset, subs in zip(range(len(all_categories)), all_categories):
#         pts = subs[cat]
#         # spread points sideways around the box center (box half-width is 0.5)
#         x = box_x(i, offset) + rng.uniform(-JITTER_HALF_LENGTH, JITTER_HALF_LENGTH, size=len(pts))
#         point_colors = np.where(pts["is_blocked"] == 1, "crimson", "gray")
#         ax.scatter(x, pts["time"], c=point_colors, s=6, alpha=0.5, zorder=3)

# labels = []
# # iterate over cat, then run
# for cat in categories:
#     label = cat
#     for name, reward in zip(names, category_rewards):
#         label = f"{label}\n{name}:{reward[cat]:.2f}"
#     labels.append(label)

# ax.set_xticks([box_x(i, (num_runs - 1) / 2) for i in range(len(categories))])
# ax.set_xticklabels(labels, fontsize=8)

# ax.set_ylabel("Measurement Step")
# ax.set_title("Iran: When Each Category was Tried", pad=22)

# # categories before this index are the "top" group, the rest are "bottom"
# split = num_categories // 2
# # halfway between the last box of the previous group and the first box of the next
# divider_x = (box_x(split - 1, num_runs - 1) + box_x(split, 0)) / 2
# ax.axvline(divider_x, color="black", linestyle="--", linewidth=1, alpha=0.5, zorder=0)

# # Below is code to add labels to each side on the top
# # can comment out and remove title padding if space needed
# # x-range covered by the boxes on each side of the divider
# left_center = (box_x(0, 0) + box_x(split - 1, num_runs - 1)) / 2
# right_center = (box_x(split, 0) + box_x(num_categories - 1, num_runs - 1)) / 2
# # x in data units, y in axes units (1.0 = top edge of the plot)
# xform = ax.get_xaxis_transform()
# ax.text(left_center, 1.0, "Most censored", transform=xform,
#         ha="center", va="bottom", fontsize=10, fontweight="bold")
# ax.text(right_center, 1.0, "Less censored", transform=xform,
#         ha="center", va="bottom", fontsize=10, fontweight="bold")

# handles = [Patch(color=color, label=name) for color, name in zip(colors, names)]
# handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor="crimson", label="blocked"))
# handles.append(Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", label="not blocked"))
# ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.0, 1.2))
# ax.spines[["top", "right"]].set_visible(False)
# plt.show()

## Heat map

In [37]:
# this looks a little different because I made Claude do this without
# the context from the rest of the notebook
TOP_N = 5

# Cells with fewer than this many measured domains are drawn gray (from Paper_results.ipynb).
# Set to 1 to only gray out cells that were never tested.
MIN_MEASURED_DOMAINS = 1

from matplotlib.patches import Patch

def add_low_count_legend(fig):
    fig.legend(handles=[Patch(facecolor="lightgray", edgecolor=SPINE_COLOR,
                              label=f"Fewer than {MIN_MEASURED_DOMAINS} measured domains")],
               loc="upper center", bbox_to_anchor=(.5, 1.0), fontsize=PAPER_LEGEND_SIZE)

countries = ["China", "Germany", "Iran", "Thailand", "Turkey",
             "United Arab Emirates", "Vietnam"]

In [38]:
def count_by_category(column_files):
    blocked, tested = {}, {}
    for label, path in column_files.items():
        if USE_FRACTIONAL_BLOCKING:
            grouped = load_run(path).groupby("category")["rewards"]
        else:
            grouped = load_run(path).groupby("category")["is_blocked"]
        blocked[label] = grouped.sum()    # how many measurements were blocked OR sum of fractional blocks
        tested[label] = grouped.count()   # how many measurements were made
    # one row per category, one column per label; 0 where a category was never tried
    if USE_FRACTIONAL_BLOCKING:
        blocked = pd.DataFrame(blocked).fillna(0).astype(float)
        blocked = round(blocked, 2)
    else:
        blocked = pd.DataFrame(blocked).fillna(0).astype(int)
    tested = pd.DataFrame(tested).fillna(0).astype(int)
    return blocked, tested

In [39]:
def plot_heatmap(column_files, exp_name, xaxis="Country"):
    blocked, tested = count_by_category(column_files)
    
    # rank categories by total blocked measurements across ALL columns
    top_categories = blocked.sum(axis=1).nlargest(TOP_N).index
    
    # ends up being ordered by count of blocked domains across all countries
    # call sort_index() on each of these to sort alphabetically instead
    blocked_top = blocked.loc[top_categories]
    tested_top = tested.loc[top_categories]
    
    # block rate; NaN where fewer than MIN_MEASURED_DOMAINS were tested, so those cells stay gray
    rate = blocked_top / tested_top.where(tested_top >= MIN_MEASURED_DOMAINS)

    fig, ax = plt.subplots(figsize=(FULL_WIDTH, 4.8))

    cmap = plt.get_cmap("YlOrRd").copy()
    cmap.set_bad("lightgray")   # untested cells show as gray
    
    im = ax.imshow(np.ma.masked_invalid(rate.to_numpy()),
                   cmap=cmap, vmin=0, vmax=1, aspect="auto")
    
    # axis labels come straight from the table, so any new columns appear automatically
    ax.set_xticks(range(rate.shape[1]))
    # wrap long column names (e.g. "United Arab Emirates") so they don't overlap at the larger font size
    ax.set_xticklabels([tex(textwrap.fill(str(c), width=10)) for c in rate.columns])
    ax.set_yticks(range(rate.shape[0]))
    ax.set_yticklabels([tex(c) for c in rate.index])
    
    format_axes(ax)
    ax.grid(False)   # format_axes turns on dotted gridlines; not wanted on a heatmap

    # thin white lines between cells (minor ticks sit on the cell borders)
    ax.set_xticks(np.arange(-0.5, rate.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rate.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.8)
    ax.tick_params(which="minor", length=0)
    
    for r in range(rate.shape[0]):
        for c in range(rate.shape[1]):
            b = blocked_top.iat[r, c]
            t = tested_top.iat[r, c]
            value = rate.iat[r, c]
    
            pct = "n/a" if np.isnan(value) else f"{value:.0%}"
            color = "white" if value > 0.6 else "black"   # readable on dark cells
    
            ax.text(c, r, f"{b}/{t}",
            # ax.text(c, r, f"{b}/{t}\n({pct})",
                    ha="center", va="center", fontsize=6, color=color)
    
    cbar = fig.colorbar(im, ax=ax, shrink=0.6)
    cbar.set_label("Block Rate")
    format_axes(cbar.ax)
    cbar.ax.grid(False)
    
    ax.set_title(bold(tex(f"Blocking Rate by Category × {xaxis}, {exp_name}")), fontweight="bold")
    add_low_count_legend(fig)
    plt.tight_layout(rect=[0, 0, 1, 0.93])

In [40]:
HEATMAP_DIR = "./figures/category_rates"

COUNTRY_CODE_MAP = {
    "China" : "CN", 
    "Germany" : "DE", 
    "Iran" : "IR", 
    "Thailand" : "TH", 
    "Turkey" : "TR",
    "United Arab Emirates" : "AE", 
    "Vietnam" : "VN"
}


In [41]:
# {column label -> CSV file}. Each entry becomes one column in the heatmap.
column_files_exp1 = {c: f"{DATA_DIR}/exp1/{c.replace(' ', '_')}.csv" for c in countries}
column_files_exp1 = {COUNTRY_CODE_MAP[k]: v for k, v in column_files_exp1.items()}
plot_heatmap(column_files_exp1, "Naive UCB")
plt.savefig(f"{HEATMAP_DIR}/countries_naiveucb.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/3745105101.py:6: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


## ASN

In [42]:
votes = pd.read_csv(f"{DATA_DIR}/exp6_asns/asn_votes_exp6.csv")

# A domain counts as blocked in an ASN if at least this fraction of that
# ASN's votes saw blocking. 0.5 = majority.
BLOCK_THRESHOLD = 0.5
votes["is_blocked"] = votes["vote_fraction"] >= BLOCK_THRESHOLD

In [43]:
def count_by_category_asn(votes, country):
    sub = votes[votes["country"] == country]
    if USE_FRACTIONAL_BLOCKING:
        grouped = sub.groupby(["category", "asn"])["vote_fraction"]
    else:
        grouped = sub.groupby(["category", "asn"])["is_blocked"]
    blocked = grouped.sum().unstack(fill_value=0)               # domains blocked
    if USE_FRACTIONAL_BLOCKING:
        blocked = round(blocked, 2)    
    else:
        blocked = blocked.astype(int)
    tested = grouped.count().unstack(fill_value=0)              # domains tested
    return blocked, tested

In [44]:
def select_top(blocked, tested, top_n=20):
    blocked = blocked[blocked.sum(axis=1) > 0]
    
    # total blocked domains per category, summed across ALL of the country's ASNs
    total_blocked = blocked.sum(axis=1)
    top_categories = total_blocked.nlargest(top_n).index   # highest -> lowest

    # .loc with this index keeps the ranking order (no re-sorting)
    blocked_top = blocked.loc[top_categories]
    tested_top = tested.loc[top_categories]

    # ASNs with the most blocking first, as before
    asn_order = blocked_top.sum(axis=0).sort_values(ascending=False).index
    blocked_top, tested_top = blocked_top[asn_order], tested_top[asn_order]

    rate = blocked_top / tested_top.where(tested_top >= MIN_MEASURED_DOMAINS)
    return blocked_top, tested_top, rate

In [45]:
def draw_grid(rate, small):
    fig, ax = plt.subplots(figsize=(FULL_WIDTH, 4.2))
    cmap = plt.get_cmap("YlOrRd").copy()
    cmap.set_bad("lightgray")
    im = ax.imshow(np.ma.masked_invalid(rate.to_numpy()),
                   cmap=cmap, vmin=0, vmax=1, aspect="auto")
    format_axes(ax)
    ax.grid(False)   # format_axes turns on dotted gridlines; not wanted on a heatmap

    ax.set_yticks(range(rate.shape[0]))
    ax.set_yticklabels([tex(c) for c in rate.index])

    
    if small:   # few ASNs: label each column and draw white gridlines
        # ax.set_xticks(range(rate.shape[1]))
        # ax.set_xticklabels(rate.columns, rotation=30, ha="right")
        # ax.set_xticks(np.arange(-0.5, rate.shape[1], 1), minor=True)
        # ax.set_yticks(np.arange(-0.5, rate.shape[0], 1), minor=True)
        # ax.grid(which="minor", color="white", linewidth=1)
        # ax.tick_params(which="minor", length=0)
        ax.set_xticks([])
    else:
        ax.set_xticks([])
    # if small:   # few ASNs: label each column and draw white gridlines
    #     ax.set_xticks(range(rate.shape[1]))
    #     ax.set_xticklabels(rate.columns, rotation=30, ha="right")
    #     ax.set_xticks(np.arange(-0.5, rate.shape[1], 1), minor=True)
    #     ax.set_yticks(np.arange(-0.5, rate.shape[0], 1), minor=True)
    #     ax.grid(which="minor", color="white", linewidth=1)
    #     ax.tick_params(which="minor", length=0)
    # else:       # many ASNs: no per-column labels
        # ax.set_xticks([])

    ax.set_xlabel(f"ASNs (n = {rate.shape[1]})")
    return fig, ax, im

In [46]:
def annotate_cells(ax, blocked_top, tested_top, rate):
    for r in range(rate.shape[0]):
        for c in range(rate.shape[1]):
            b, t, value = blocked_top.iat[r, c], tested_top.iat[r, c], rate.iat[r, c]
            pct = "n/a" if np.isnan(value) else f"{value:.0%}"
            color = "white" if value > 0.6 else "black"
            # ax.text(c, r, f"{b}/{t}\n({pct})",
            ax.text(c, r, f"{b}/{t}",
                    ha="center", va="center", fontsize=9, color=color)

In [47]:
SMALL_MAX_COLUMNS = 15   # at or below this, label and annotate every ASN

def create_asn_heatmap(votes, country, top_n=20):
    blocked, tested = count_by_category_asn(votes, country)
    blocked_top, tested_top, rate = select_top(blocked, tested, top_n)

    small = rate.shape[1] <= SMALL_MAX_COLUMNS
    # small = False
    fig, ax, im = draw_grid(rate, small)
    # if small:
    #     annotate_cells(ax, blocked_top, tested_top, rate)

    cbar = fig.colorbar(im, ax=ax, shrink=0.6)
    cbar.set_label("Block Rate")
    format_axes(cbar.ax)
    cbar.ax.grid(False)
    ax.set_title(bold(tex(f"Blocking Rate by Category × ASN: {country}")), fontweight="bold")
    add_low_count_legend(fig)
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    

In [48]:
create_asn_heatmap(votes, "Iran")
plt.savefig(f"{HEATMAP_DIR}/asns_iran.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/97208187.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [49]:
create_asn_heatmap(votes, "China")
plt.savefig(f"{HEATMAP_DIR}/asns_china.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/2231145071.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [50]:
create_asn_heatmap(votes, "United Arab Emirates")
plt.savefig(f"{HEATMAP_DIR}/asns_uae.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/3922427769.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [51]:
create_asn_heatmap(votes, "Germany")
plt.savefig(f"{HEATMAP_DIR}/asns_germany.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/532796063.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [52]:
create_asn_heatmap(votes, "Thailand")
plt.savefig(f"{HEATMAP_DIR}/asns_thailand.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/4080243452.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [53]:
create_asn_heatmap(votes, "Turkey")
plt.savefig(f"{HEATMAP_DIR}/asns_turkey.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/3760672216.py:2: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(FULL_WIDTH, 4.2))
/tmp/ipykernel_56799/3101326595.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [54]:
create_asn_heatmap(votes, "Vietnam")
plt.savefig(f"{HEATMAP_DIR}/asns_vietnam.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

/tmp/ipykernel_56799/1451072735.py:3: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


In [74]:
from matplotlib.transforms import blended_transform_factory

ASN_PANEL_COUNTRIES = [("China", "China"), ("Iran", "Iran"), ("United Arab Emirates", "UAE")]

USE_FRACTIONAL_BLOCKING = True

categories = []

def plot_asn_panels(votes, exp_name, top_n=5, min_domains=50, categories_in=None,
                    figsize=(FULL_WIDTH, 3.0)):
    """China / Iran / UAE category x ASN heatmaps in ONE figure.
    The same top_n categories are used in every panel (the ones with the most blocking
    across the three countries, among categories with at least min_domains domains tested
    in every country), so the row labels appear once, on the left."""
    title=name_map[exp_name]

    votes = pd.read_csv(f"{DATA_DIR}/{exp_name}_asns/asn_votes_{exp_name}.csv")
    votes["is_blocked"] = votes["vote_fraction"] >= BLOCK_THRESHOLD

    
    # --- 1. per-country counts, and the shared set of categories ---
    counts = {c: count_by_category_asn(votes, c) for c, _ in ASN_PANEL_COUNTRIES}
    # domains tested per category in each country (the largest count across that country's ASNs);
    # a category missing from a country counts as 0
    totals = pd.concat([blocked.sum(axis=1) for blocked, _ in counts.values()]).groupby(level=0).sum()

    if categories_in is None:
        n_tested = pd.concat([tested.max(axis=1) for _, tested in counts.values()], axis=1).fillna(0)
        eligible = n_tested.index[(n_tested >= min_domains).all(axis=1)]   # enough domains in ALL countries
        categories = totals[totals.index.isin(eligible)].nlargest(top_n).index   # highest -> lowest
        if len(categories) == 0:
            raise ValueError(f"No category has at least {min_domains} domains tested in every country.")
    else:
        # your own list: plotted in the order given; top_n and min_domains are ignored
        categories = pd.Index(categories_in)
        print(categories_in)
        missing = [c for c in categories if c not in totals.index]
        if missing:
            raise ValueError(f"Categories not found in the votes data: {missing}")

    fig, axes = plt.subplots(1, len(ASN_PANEL_COUNTRIES), figsize=figsize)
    cmap = plt.get_cmap("YlOrRd").copy()
    # cmap.set_bad("lightgray")                          # fewer than MIN_MEASURED_DOMAINS

    # need to special case for just one country
    if len(ASN_PANEL_COUNTRIES) == 1:
        i, ax, (country, short) = 0, axes, ASN_PANEL_COUNTRIES[0]
        blocked, tested = counts[country]
        blocked = blocked.reindex(categories).fillna(0)
        tested = tested.reindex(categories).fillna(0)
        asn_order = blocked.sum(axis=0).sort_values(ascending=False).index
        blocked, tested = blocked[asn_order], tested[asn_order]
        rate = 100 * blocked / tested.where(tested >= MIN_MEASURED_DOMAINS)   # percent

        # --- 3. heatmap ---
        im = ax.imshow(np.ma.masked_invalid(rate.to_numpy()), cmap=cmap, vmin=0, vmax=100, aspect="auto")
        format_axes(ax)
        ax.grid(False)
        ax.set_xticks([])                              # no per-ASN labels
        ax.tick_params(axis="y", length=0)
        ax.set_title(tex(f"UCB with ASN fraction reward, {short} ({rate.shape[1]} ASNs)"), fontsize=14)
        if i == 0:
            ax.set_yticks(range(len(categories)))
            ax.set_yticklabels([tex(textwrap.fill(c, width=16)) for c in categories])
        else:
            ax.set_yticks([])

        # --- 4. right-hand "n" column: domains measured per category (range across ASNs) ---
        trans = blended_transform_factory(ax.transAxes, ax.transData)   # x: axes fraction, y: row number
        for r in range(len(categories)):
            # n_per_asn = tested.iloc[r][tested.iloc[r] > 0]
            # print(blocked)
            # if n_per_asn.empty:
            #     continue
            # lo, hi = int(n_per_asn.min()), int(n_per_asn.max())
            # ax.text(1.04, r, f"{lo}" if lo == hi else f"{lo}-{hi}", transform=trans, ha="left", va="center")

            blocked_per_asn = blocked.iloc[r][blocked.iloc[r] > 0]
            print(blocked_per_asn)
            percent = np.mean(blocked_per_asn) / 2000 * 100
            if np.isnan(percent):
                percent = 0
            ax.text(1.04, r, f"{percent:.2f}\%", transform=trans, ha="left", va="center", fontsize=12)
            
        ax.text(1.04, 1.02, "Block \%", transform=ax.transAxes, ha="left", va="bottom", fontsize=12)

    else:
        for i, (ax, (country, short)) in enumerate(zip(axes, ASN_PANEL_COUNTRIES)):
            # --- 2. rows = shared categories, columns = this country's ASNs (most blocking first) ---
            blocked, tested = counts[country]
            blocked = blocked.reindex(categories).fillna(0)
            tested = tested.reindex(categories).fillna(0)
            asn_order = blocked.sum(axis=0).sort_values(ascending=False).index
            blocked, tested = blocked[asn_order], tested[asn_order]
            rate = 100 * blocked / tested.where(tested >= MIN_MEASURED_DOMAINS)   # percent
    
            # --- 3. heatmap ---
            im = ax.imshow(np.ma.masked_invalid(rate.to_numpy()), cmap=cmap, vmin=0, vmax=100, aspect="auto")
            format_axes(ax)
            ax.grid(False)
            ax.set_xticks([])                              # no per-ASN labels
            ax.tick_params(axis="y", length=0)
            ax.set_title(tex(f"{short} ({rate.shape[1]} ASNs)"))
            if i == 0:
                ax.set_yticks(range(len(categories)))
                ax.set_yticklabels([tex(textwrap.fill(c, width=16)) for c in categories])
            else:
                ax.set_yticks([])
    
            # --- 4. right-hand "n" column: domains measured per category (range across ASNs) ---
            trans = blended_transform_factory(ax.transAxes, ax.transData)   # x: axes fraction, y: row number
            for r in range(len(categories)):
                # n_per_asn = tested.iloc[r][tested.iloc[r] > 0]
                # print(blocked)
                # if n_per_asn.empty:
                #     continue
                # lo, hi = int(n_per_asn.min()), int(n_per_asn.max())
                # ax.text(1.04, r, f"{lo}" if lo == hi else f"{lo}-{hi}", transform=trans, ha="left", va="center")
    
                blocked_per_asn = blocked.iloc[r][blocked.iloc[r] > 0]
                print(blocked_per_asn)
                percent = np.mean(blocked_per_asn) / 2000 * 100
                if np.isnan(percent):
                    percent = 0
                ax.text(1.04, r, f"{percent:.2f}\%", transform=trans, ha="left", va="center", fontsize=14)
                
            ax.text(1.04, 1.02, "Block \%", transform=ax.transAxes, ha="left", va="bottom", fontsize=14)

    # --- 5. figure title, margins (inches) and the shared horizontal color bar ---
    # fig.suptitle(tex(title), fontsize=PAPER_PANEL_LABEL_SIZE, y=0.99)
    set_margins(fig, left=1.0, right=0.4, bottom=0.7, top=0.55)
    fig.subplots_adjust(wspace=0.35)

    w, h = fig.get_size_inches()
    cax = fig.add_axes([0.3, 0.45 / h, 0.4, 0.12 / h])        # [left, bottom, width, height] as figure fractions
    cbar = fig.colorbar(im, cax=cax, orientation="horizontal")
    cbar.set_ticks([0, 50, 100])
    cbar.set_label(tex("Mean VP blocked-vote fraction (%)" if USE_FRACTIONAL_BLOCKING
                       else "Domains blocked (%)"), fontsize=12)
    format_axes(cax)
    cax.grid(False)
    return fig, axes


<>:83: SyntaxWarning: invalid escape sequence '\%'
<>:85: SyntaxWarning: invalid escape sequence '\%'
<>:125: SyntaxWarning: invalid escape sequence '\%'
<>:127: SyntaxWarning: invalid escape sequence '\%'
<>:83: SyntaxWarning: invalid escape sequence '\%'
<>:85: SyntaxWarning: invalid escape sequence '\%'
<>:125: SyntaxWarning: invalid escape sequence '\%'
<>:127: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_56799/3601165216.py:83: SyntaxWarning: invalid escape sequence '\%'
  ax.text(1.04, r, f"{percent:.2f}\%", transform=trans, ha="left", va="center", fontsize=12)
/tmp/ipykernel_56799/3601165216.py:85: SyntaxWarning: invalid escape sequence '\%'
  ax.text(1.04, 1.02, "Block \%", transform=ax.transAxes, ha="left", va="bottom", fontsize=12)
/tmp/ipykernel_56799/3601165216.py:125: SyntaxWarning: invalid escape sequence '\%'
  ax.text(1.04, r, f"{percent:.2f}\%", transform=trans, ha="left", va="center", fontsize=14)
/tmp/ipykernel_56799/3601165216.py:127: SyntaxWarning: in

In [ ]:
# plot_asn_panels(votes, "exp5", categories_in=["Video Streaming", "Photography", "Pornography", "Social Networks", "Television"])
# plt.savefig(f"{HEATMAP_DIR}/results_asn_blocking_vp_vote.pdf", format="pdf", bbox_inches='tight', dpi=300)
# plot_asn_panels(votes, "exp6", categories_in=["Video Streaming", "Photography", "Pornography", "Social Networks", "Television"])
# plt.savefig(f"{HEATMAP_DIR}/results_asn_blocking_asn_vote.pdf", format="pdf", bbox_inches='tight', dpi=300)
# plot_asn_panels(votes, "exp2", categories_in=["Video Streaming", "Photography", "Pornography", "Social Networks", "Television"])
# plt.savefig(f"{HEATMAP_DIR}/results_asn_blocking_linucb.pdf", format="pdf", bbox_inches='tight', dpi=300)

In [75]:
ASN_PANEL_COUNTRIES = [("China", "China")]
USE_FRACTIONAL_BLOCKING = True

plot_asn_panels(votes, "exp5", categories_in=["Video Streaming", "Photography", "Pornography", "Social Networks", "Television"])
plt.savefig(f"{HEATMAP_DIR}/results_china_asn_blocking_vp_vote.pdf", format="pdf", bbox_inches='tight', dpi=300)
plot_asn_panels(votes, "exp6", categories_in=["Video Streaming", "Photography", "Pornography", "Social Networks", "Television"])
plt.savefig(f"{HEATMAP_DIR}/results_china_asn_blocking_asn_vote.pdf", format="pdf", bbox_inches='tight', dpi=300)
plot_asn_panels(votes, "exp2", categories_in=["Video Streaming", "Photography", "Pornography", "Social Networks", "Television"])
plt.savefig(f"{HEATMAP_DIR}/results_china_asn_blocking_linucb.pdf", format="pdf", bbox_inches='tight', dpi=300)

['Video Streaming', 'Photography', 'Pornography', 'Social Networks', 'Television']
asn
asn:24362      79.00
asn:24364     128.00
asn:55990     115.86
asn:137695    125.00
asn:135061    121.00
               ...  
asn:134774     84.00
asn:58811      92.00
asn:38375      86.00
asn:9308       70.17
asn:139141     56.50
Name: Video Streaming, Length: 79, dtype: float64
asn
asn:24362     29.00
asn:24364     46.00
asn:55990     40.29
asn:137695    39.00
asn:135061    43.00
              ...  
asn:134774    30.50
asn:58811     20.00
asn:38375     27.00
asn:9308      21.67
asn:139141    20.50
Name: Photography, Length: 79, dtype: float64
asn
asn:24362     198.00
asn:24364      70.00
asn:55990      34.00
asn:137695     25.00
asn:135061     24.00
               ...  
asn:134774     15.50
asn:58811      14.00
asn:38375      11.00
asn:9308        5.33
asn:139141     11.50
Name: Pornography, Length: 79, dtype: float64
asn
asn:24364     2.00
asn:55990     1.71
asn:45090     1.53
asn:17622     0.10
a

In [ ]:
# {column label -> CSV file}. Each entry becomes one column in the heatmap.
column_files_exp3 = {c: f"{DATA_DIR}/exp3/{c.replace(' ', '_')}.csv" for c in countries}
plot_heatmap(column_files_exp3, "LinUCB")
plt.savefig(f"{HEATMAP_DIR}/countries_linucb.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# {column label -> CSV file}. Each entry becomes one column in the heatmap.
column_files_exp7 = {c: f"{DATA_DIR}/exp7/{c.replace(' ', '_')}.csv" for c in countries}
plot_heatmap(column_files_exp7, "UCB Partial (ASNs)", "ASNs")

In [ ]:
# {column label -> CSV file}. Each entry becomes one column in the heatmap.
column_files_baseline = {c: f"{DATA_DIR}/baseline/{c.replace(' ', '_')}.csv" for c in countries}
plot_heatmap(column_files_baseline, "Baseline")
plt.savefig(f"{HEATMAP_DIR}/countries_baseline.pdf", format="pdf", bbox_inches='tight', dpi=300)
plt.show()

# oneoff

In [ ]:
import ast
from matplotlib.ticker import MaxNLocator

# Adjust to wherever you keep the category inventory (the same file Paper_results.ipynb reads).
INVENTORY_PATH = f"{DATA_DIR}/tranco_categories_subdomain_tld_entities_top10k.csv"

IRAN_CONFIGS = ["baseline", "exp5", "exp2", "exp3", "exp4", "exp6", "exp7"]
IRAN_CONFIG_LABELS = ["Ranked", "Default", "Lin", "Shared", "Cluster", "ASN", "Maj"]
TOPICS = ["Pornography", "Social Networks", "News & Media", "Video Streaming", "Ecommerce"]

# --- 1. inventory: how many candidate domains each category has ---
inventory = pd.read_csv(INVENTORY_PATH, sep="|")
inventory["categories"] = inventory["categories"].map(ast.literal_eval).map(
    lambda x: x if x else ["Newly Seen Domains"])            # a domain can belong to several categories
candidate_sizes = (inventory[["domain", "categories"]].explode("categories")
                   .groupby("categories")["domain"].nunique().rename("candidates"))

# --- 2. all Iran runs in one table ---
runs = []
for config in IRAN_CONFIGS:
    run = pd.read_csv(f"{DATA_DIR}/{config}/Iran.csv")
    run["config"] = config
    runs.append(run)
runs = pd.concat(runs, ignore_index=True)
runs["arm"] = runs["action"].str.replace("categories ", "", regex=False)

# --- 3. panel (a) data: how often each config picked each category ---
allocation_matrix = (runs.groupby(["arm", "config"]).size().unstack("config")
                     .reindex(index=TOPICS, columns=IRAN_CONFIGS).fillna(0))

# --- 4. panel (b) data + table: Default (exp5) coverage of each category's domains ---
default_run = runs[runs["config"] == "exp5"].merge(
    inventory[["domain", "categories"]], left_on="targets", right_on="domain", how="left")
membership = default_run.explode("categories")               # one row per (measured domain, category it belongs to)
coverage_matrix = membership.groupby("categories").agg(
    measured=("targets", "nunique"),
    fractional_blocking=("rewards", "sum"),                  # sum of fractional rewards instead of the 0/1 flag
).join(candidate_sizes)
coverage_matrix["inventory_coverage"] = coverage_matrix["measured"] / coverage_matrix["candidates"]
coverage_matrix = coverage_matrix.reindex(TOPICS)
coverage_matrix.index.name = "categories"

# --- 5. figure ---
fig, axes = plt.subplots(1, 2, figsize=(FULL_WIDTH, 2.6), gridspec_kw={"width_ratios": [1.25, 1]})

im = axes[0].imshow(allocation_matrix.to_numpy(), cmap="Blues", aspect="auto", vmin=0, vmax=700)
for i in range(len(TOPICS)):
    for j in range(len(IRAN_CONFIGS)):
        value = int(allocation_matrix.iloc[i, j])
        axes[0].text(j, i, str(value), ha="center", va="center",
                     fontsize=7, color="white" if value > 400 else "black")
axes[0].set(xticks=np.arange(len(IRAN_CONFIGS)), yticks=np.arange(len(TOPICS)),
            yticklabels=[tex(t) for t in TOPICS])
axes[0].set_xticklabels([bold(l) for l in IRAN_CONFIG_LABELS], rotation=45, ha="right")
for tick, config in zip(axes[0].get_xticklabels(), IRAN_CONFIGS):
    tick.set_color(box_color_map[config])
    tick.set_fontweight("bold")
axes[0].set_title("(a) Iran: category selections", loc="left")

x = np.arange(len(TOPICS))
axes[1].barh(x, coverage_matrix["candidates"], color="#dddddd", height=.7, label="Candidates")
axes[1].barh(x, coverage_matrix["measured"], color=box_color_map["exp5"], height=.45, label="Measured members")
axes[1].set(yticks=x, yticklabels=[], xlabel="Distinct domain members")
axes[1].invert_yaxis()
axes[1].xaxis.set_major_locator(MaxNLocator(3))
axes[1].set_axisbelow(True)
axes[1].set_title("(b) Default: coverage", loc="left")
axes[1].legend(loc="lower center", bbox_to_anchor=(.5, 1.20), ncol=2,
               columnspacing=.8, handlelength=1.2)

for ax in axes:
    format_axes(ax)
axes[0].grid(False)           # the cell colors already encode the counts
axes[1].yaxis.grid(False)
fig.tight_layout(pad=.6, w_pad=1)
plt.savefig("temp.pdf")

# --- 6. table (fractional blocking replaces the old 0/1 "flagged" column) ---
table = coverage_matrix[["measured", "candidates", "fractional_blocking", "inventory_coverage"]].round(
    {"fractional_blocking": 2, "inventory_coverage": 3})
display(table)

## Arm selection over time

In [ ]:
NUM_ARMS_TO_COMPARE = 10

exp = load_run(f"{DATA_DIR}/exp1/Iran.csv")

top_arms = exp["category"].value_counts().head(NUM_ARMS_TO_COMPARE).index
print(list(top_arms))  # check which categories were picked

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for cat in top_arms:
    # True/False per measurement -> running total of times this arm was picked
    ax.plot(exp["time"], (exp["category"] == cat).cumsum(), label=cat)

ax.set_title(f"Iran (exp6): Top {NUM_ARMS_TO_COMPARE} Categories Selected Over Time")
format_axes(ax)
ax.set_axisbelow(True)
fig.legend(*ax.get_legend_handles_labels(), ncol=2, loc="upper center", bbox_to_anchor=(.5, 1.0),
           frameon=True, edgecolor="#cccccc", borderpad=.3)
fig.supxlabel("Measurement", y=.02, fontsize=PAPER_AXIS_LABEL_SIZE)
fig.supylabel("Cumulative Attempts", x=.006, fontsize=PAPER_AXIS_LABEL_SIZE)
fig.subplots_adjust(left=.14, right=.97, bottom=.2, top=.5)
plt.show()